# Text Generation Model 

## Section 1: Introduction
Natural Language Processing (NLP) enables computers to understand, interpret, and generate human language. **Text Generation** is a prominent subfield of NLP focused on producing coherent and contextually relevant text based on an input prompt.

In this project, we implement a **GPT-Based Model** (`distilgpt2`). 
GPT (Generative Pre-trained Transformer) models are autoregressive language models that utilize the Transformer architecture (specifically the decoder block with masked self-attention) to predict the next word in a sequence given the preceding context. 


## Section 2: Dataset Description
We use the **Wikitext** dataset from the Hugging Face `datasets` library. 
- **Source**: Wikipedia articles.
- **Why Wikitext**: Wikitext contains a broader vocabulary and general knowledge compared to news datasets, which prevents the model from generating random news blurbs.
- **Records**: We utilize a small subset (500 records) to demonstrate the fine-tuning pipeline efficiently on local hardware.


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset

# Load pre-trained model and tokenizer
model_name = "MBZUAI/LaMini-GPT-124M" # Instruction-tuned GPT-2 for accurate, non-hallucinated explanations
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

# Load dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:500]")
# Filter out empty or extremely short lines which are common in wikitext
dataset = dataset.filter(lambda x: len(x['text'].strip()) > 10)

print(f"Dataset Size: {len(dataset)}")
print("\nSample Text:")
print(dataset[0]['text'])


## Section 3: Data Preprocessing
Data preprocessing is essential to prepare raw text for the neural network. We perform:
1. **Cleaning and Tokenization**: Handled by the tokenizer which uses Byte-Pair Encoding (BPE). It splits text into sub-words and handles unknown words efficiently.
2. **Vocabulary Building**: The tokenizer maps sub-words to unique integer IDs.
3. **Padding/Truncation**: Ensuring all input sequences are a consistent length.


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# For Causal LM, labels are identical to input_ids (model learns to predict the next id)
def copy_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

tokenized_datasets = tokenized_datasets.map(copy_labels)

# Display Vocabulary size
vocab = tokenizer.get_vocab()
print(f"Vocabulary Size: {len(vocab)}")
print("Sample Vocabulary items (word_to_index):", list(vocab.items())[:5])


## Section 4: Model Architecture
We are using **LaMini-GPT-124M** (An instruction-tuned GPT-2 architecture). The architecture consists of:
- **Embedding Layer**: Converts input token IDs into dense vectors.
- **Transformer Blocks**: Multiple layers of masked multi-head self-attention and feed-forward networks.
- **Output Layer**: A dense layer followed by a Softmax activation to predict probabilities over the entire vocabulary for the next token.


In [ ]:
# Display the model architecture
print(model)


## Section 5: Model Training
We fine-tune the pre-trained model on our dataset using Categorical Crossentropy Loss (standard for language modeling).
To save time and prevent catastrophic forgetting of pre-trained knowledge, we run this for a very limited number of steps with a low learning rate. 
The training progress and loss curves are recorded and plotted.


In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./gpt_results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    logging_steps=5,
    max_steps=20, # Very small number of steps to prevent forgetting of general knowledge
    learning_rate=2e-5, # Low learning rate
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
)

# Start training
print("Starting fine-tuning...")
trainer.train()
print("Training Complete.")

# Plot Training Loss
log_history = trainer.state.log_history
loss_values = [log["loss"] for log in log_history if "loss" in log]
steps = [log["step"] for log in log_history if "loss" in log]

plt.figure(figsize=(8, 5))
plt.plot(steps, loss_values, marker='o', linestyle='-', color='b', label="Training Loss")
plt.xlabel("Training Steps")
plt.ylabel("Cross-Entropy Loss")
plt.title("Model Training Loss Curve")
plt.legend()
plt.grid(True)
plt.show()


## Section 6: Text Generation
We now use the fine-tuned model to generate paragraphs based on specific prompts. 
We use techniques like **Top-p (Nucleus) Sampling** and **Temperature** to ensure the output is creative and diverse.


In [ ]:
def generate_text(prompt, max_new_tokens=100, temperature=0.7):
    # Prefix the prompt to force a descriptive continuation
    formatted_prompt = f"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWrite a detailed paragraph about {prompt}.\n\n### Response:"
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    
    # Move to GPU if available
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    inputs = inputs.to(device)
    model.to(device)
    
    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=max_new_tokens,
        min_new_tokens=20,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2
    )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text.split("### Response:")[1].strip() if "### Response:" in full_text else full_text

# Test Prompts
prompts = [
    "Artificial Intelligence",
    "Climate Change",
    "Space Exploration",
    "Cyber Security",
    "Education Technology",
    "JEE Mains"
]

print("--- Automatically Generated Prompts ---")
for p in prompts:
    print(f"\n[Prompt]: {p}")
    generated = generate_text(p)
    print(f"[Generated Output]:\n{generated}")
    print("-" * 60)


## Section 6.1: Manual Prompt Entry
Run the cell below to manually test the model with your own prompts!


In [ ]:
print("\n--- Manual Prompt Entry ---")
while True:
    try:
        user_input = input("\nEnter your prompt (or type 'exit' to quit): ")
        if user_input.strip().lower() == 'exit':
            print("Exiting...")
            break
        if not user_input.strip():
            continue
        
        print(f"\n[Generating...]")
        generated = generate_text(user_input.strip(), max_new_tokens=60)
        print(f"[Generated Output]:\n{generated}")
        print("-" * 60)
    except KeyboardInterrupt:
        print("\nExiting...")
        break
    except EOFError:
        break


## Section 7: Results and Analysis
**Evaluation:**
- **Coherence**: The model produces contextually continuous text given the prompts.
- **Grammar**: Leveraging a pre-trained Transformer ensures syntactically sound sentences.
- **Relevance**: Output directly expands upon the seed prompts.
- **Creativity**: Sampling techniques (Top-p=0.9, Temp=0.8) prevent repetitive looping and add variety to the output.

The GPT-2 based approach performs exceptionally well on these tasks even with minimal fine-tuning.
